# Lesson 14: Physics-Informed Neural Networks - Full Solution Notebook

Course: Machine Learning for Chemical Engineering

Focus: PDE residuals, collocation points, sparse sensing, and physics-informed loss.

## Learning goals
1. Fit sparse data with and without a physics penalty.
2. Use automatic differentiation to compute derivatives.
3. Compare the unconstrained neural network and PINN.
4. Explain why nondimensionalization and loss balancing matter.

## Colab use
Run cells from top to bottom. The notebook is self-contained and uses synthetic data so it can run from GitHub/Colab without external files.

## Full worked solution
This version contains one possible complete implementation. Use it after attempting the exercise notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(14)
k = 1.2
z_data = torch.tensor([[0.0], [0.35], [0.7], [1.0]], dtype=torch.float32)
T_data = torch.exp(-k*z_data)
z_colloc = torch.linspace(0, 1, 80).reshape(-1, 1)

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))
    def forward(self, z):
        return self.net(z)

def train(model, physics_weight=0.0, steps=2500):
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    history = []
    for _ in range(steps):
        opt.zero_grad()
        data_loss = torch.mean((model(z_data) - T_data)**2)
        physics_loss = torch.tensor(0.0)
        if physics_weight > 0:
            zc = z_colloc.clone().detach().requires_grad_(True)
            T = model(zc)
            dTdz = torch.autograd.grad(T, zc, grad_outputs=torch.ones_like(T), create_graph=True)[0]
            residual = dTdz + k*T
            physics_loss = torch.mean(residual**2)
        loss = data_loss + physics_weight*physics_loss
        loss.backward()
        opt.step()
        history.append(float(loss.detach()))
    return np.array(history)

plain = Net()
pinn = Net()
train(plain, physics_weight=0.0)
train(pinn, physics_weight=1.0)

z = torch.linspace(0, 1, 200).reshape(-1, 1)
with torch.no_grad():
    true = torch.exp(-k*z).numpy().ravel()
    pred_plain = plain(z).numpy().ravel()
    pred_pinn = pinn(z).numpy().ravel()

plt.figure(figsize=(7, 4))
plt.plot(z.numpy(), true, "k--", label="true solution")
plt.plot(z.numpy(), pred_plain, label="data-only NN")
plt.plot(z.numpy(), pred_pinn, label="PINN")
plt.scatter(z_data.numpy(), T_data.numpy(), c="k", zorder=3, label="sparse sensors")
plt.xlabel("z")
plt.ylabel("T(z)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
print("The PINN uses the ODE residual as virtual data between sensors.")

## Reflection

Use the numerical result and the plots to answer:

1. What physical assumption did the model encode?
2. Where could the model fail outside the sampled domain?
3. Which validation metric would matter most in a real engineering deployment?